# AI Assignment (2026-2027): Iris Flower Classification

## 1. Project Overview & Objectives
The **Iris Flower Classification** dataset is one of the most famous benchmark datasets in pattern recognition and machine learning, introduced by British statistician and biologist **Ronald Fisher** in 1936.

### Objectives:
1. **Exploratory Data Analysis (EDA)**: Understand distribution, correlation, and distinct boundaries between features.
2. **Data Preprocessing**: Stratified splitting and feature scaling using `StandardScaler`.
3. **Model Development**: Train and evaluate multiple machine learning algorithms:
   - Logistic Regression
   - K-Nearest Neighbors (KNN)
   - Support Vector Machine (SVM)
   - Decision Tree Classifier
   - Random Forest Ensemble
   - Multi-Layer Perceptron (Neural Network - MLP)
4. **Model Validation**: Assess generalization using 5-Fold Stratified Cross-Validation, Confusion Matrices, and Classification Reports (Precision, Recall, F1-Score).
5. **Inference**: Test custom sample predictions with probability confidence scores.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# Set style and seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries successfully imported!")

## 2. Data Loading & Inspection

In [ ]:
iris_raw = load_iris()
feature_names = [name.replace(" (cm)", "").replace(" ", "_") for name in iris_raw.feature_names]
target_names = list(iris_raw.target_names)

df = pd.DataFrame(data=iris_raw.data, columns=feature_names)
df["target"] = iris_raw.target
df["species"] = df["target"].map({i: name for i, name in enumerate(target_names)})

# Display dataset summary
print(f"Total Samples: {df.shape[0]}, Total Features: {len(feature_names)}")
print("Target Classes:", target_names)
df.head()

In [ ]:
print("--- Missing Values Check ---")
print(df.isnull().sum())

print("\n--- Class Balance ---")
print(df["species"].value_counts())

print("\n--- Descriptive Statistics ---")
df.describe().round(3)

## 3. Exploratory Data Analysis (EDA)
Visualizing feature distributions and relationships gives intuition into which features offer the highest separability.

In [ ]:
# Feature distribution histograms by species
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for idx, feature in enumerate(feature_names):
    ax = axes[idx // 2, idx % 2]
    sns.histplot(data=df, x=feature, hue="species", kde=True, ax=ax, palette="Set2", alpha=0.6)
    ax.set_title(f"Distribution of {feature.replace('_', ' ').title()}", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(7, 5))
corr = df[feature_names].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar=True)
plt.title("Correlation Matrix of Features", fontweight="bold")
plt.show()

In [ ]:
# Pairplot showing pairwise feature separability
pairplot = sns.pairplot(df[feature_names + ["species"]], hue="species", palette="Dark2")
pairplot.fig.subplots_adjust(top=0.94)
pairplot.fig.suptitle("Pairwise Relationships of Iris Features by Species", fontweight="bold")
plt.show()

## 4. Data Preprocessing
- **Train-Test Split**: 80% for training, 20% for testing with **stratification** to preserve class balance.
- **Standardization**: Scaling features using $z = \frac{x - \mu}{\sigma}$ to prevent distance-sensitive algorithms (KNN, SVM, MLP) from being dominated by scale.

In [ ]:
X = df[feature_names].values
y = df["target"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

## 5. Model Training & Cross-Validation
We evaluate 6 distinct machine learning and neural network classifiers.

In [ ]:
models = {
    "Logistic Regression": {
        "clf": LogisticRegression(max_iter=300, random_state=RANDOM_STATE),
        "scaled": True
    },
    "K-Nearest Neighbors (KNN)": {
        "clf": KNeighborsClassifier(n_neighbors=5),
        "scaled": True
    },
    "Support Vector Machine (SVM)": {
        "clf": SVC(kernel="rbf", C=1.0, probability=True, random_state=RANDOM_STATE),
        "scaled": True
    },
    "Decision Tree": {
        "clf": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
        "scaled": False
    },
    "Random Forest": {
        "clf": RandomForestClassifier(n_estimators=100, max_depth=4, random_state=RANDOM_STATE),
        "scaled": False
    },
    "Multi-Layer Perceptron (Neural Net)": {
        "clf": MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=600, random_state=RANDOM_STATE),
        "scaled": True
    }
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []
trained_models = {}
cms = {}

for name, item in models.items():
    clf = item["clf"]
    x_tr = X_train_scaled if item["scaled"] else X_train
    x_te = X_test_scaled if item["scaled"] else X_test
    
    # 5-fold cross validation on training data
    cv_scores = cross_val_score(clf, x_tr, y_train, cv=cv, scoring="accuracy")
    
    # Fit and evaluate on test set
    clf.fit(x_tr, y_train)
    y_pred = clf.predict(x_te)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    
    trained_models[name] = clf
    cms[name] = confusion_matrix(y_test, y_pred)
    
    results.append({
        "Model": name,
        "5-Fold CV Mean": round(cv_scores.mean(), 4),
        "5-Fold CV Std": round(cv_scores.std(), 4),
        "Test Accuracy": round(acc, 4),
        "Precision (Macro)": round(prec, 4),
        "Recall (Macro)": round(rec, 4),
        "F1-Score (Macro)": round(f1, 4)
    })

results_df = pd.DataFrame(results).sort_values(by="Test Accuracy", ascending=False).reset_index(drop=True)
results_df

## 6. Performance Visualizations
### A. Comparative Model Performance

In [ ]:
plt.figure(figsize=(10, 5))
x_indices = np.arange(len(results_df))
bar_width = 0.35

plt.bar(x_indices - bar_width/2, results_df["5-Fold CV Mean"], width=bar_width, label="5-Fold CV Mean Acc", color="#4C72B0")
plt.bar(x_indices + bar_width/2, results_df["Test Accuracy"], width=bar_width, label="Test Set Accuracy", color="#55A868")

plt.xlabel("Model", fontweight="bold")
plt.ylabel("Accuracy", fontweight="bold")
plt.title("Comparison of Model Accuracy (Cross-Validation vs Test Set)", fontweight="bold")
plt.xticks(x_indices, results_df["Model"], rotation=20, ha="right")
plt.ylim(0.85, 1.05)
plt.legend()
plt.tight_layout()
plt.show()

### B. Confusion Matrices for All Models

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx, (name, cm) in enumerate(cms.items()):
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=target_names, yticklabels=target_names)
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

plt.tight_layout()
plt.show()

## 7. Sample Inference & Prediction
We can test the best performing model on new unseen flower measurements to classify the species along with confidence probabilities.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_clf = trained_models[best_model_name]
is_scaled = models[best_model_name]["scaled"]
print(f"Best Selected Model: {best_model_name}")

# Custom input measurements [sepal_length, sepal_width, petal_length, petal_width]
new_samples = np.array([
    [5.0, 3.4, 1.5, 0.2],  # Expected: Setosa
    [5.9, 3.0, 4.2, 1.5],  # Expected: Versicolor
    [6.5, 3.0, 5.5, 2.0]   # Expected: Virginica
])

new_samples_input = scaler.transform(new_samples) if is_scaled else new_samples
predictions = best_clf.predict(new_samples_input)
probabilities = best_clf.predict_proba(new_samples_input) if hasattr(best_clf, "predict_proba") else None

for i, sample in enumerate(new_samples):
    pred_species = target_names[predictions[i]]
    prob_str = f" | Confidence: {probabilities[i][predictions[i]]*100:.2f}%" if probabilities is not None else ""
    print(f"Sample {i+1}: {sample.tolist()} -> Predicted: {pred_species.upper()}{prob_str}")

## 8. Conclusion & Summary
- **Feature Importance**: Petal length and petal width are the most discriminative features for separating Iris species, with Iris Setosa being completely linearly separable from Versicolor and Virginica.
- **Model Performance**: High-accuracy classifiers (SVM, KNN, Random Forest, Logistic Regression) reliably achieve 96% to 100% test accuracy on stratified splits.
- **Neural Network (MLP)**: Successfully converged and achieved high classification accuracy, demonstrating effective non-linear decision boundary mapping.